# CS3807 Deep Learning Laboratory - Part 4: Evaluation Metrics and Result Tables
Computes test accuracy / precision / recall / F1 for all 21 models, builds 7 result tables,
and plots confusion matrices for the baseline and the best model.

In [ ]:
# Cell 1: Imports, paths, experiment registry
import os, pickle, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             precision_recall_fscore_support)
os.makedirs('results/tables', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)
EXP_FILES = {'exp1': 'results/exp1_histories.pkl', 'exp2': 'results/exp2_histories.pkl',
             'exp3': 'results/exp3_histories.pkl', 'exp4': 'results/exp4_histories.pkl',
             'exp5': 'results/exp5_histories.pkl', 'additional': 'results/additional_histories.pkl'}
EXP_MODELS = {'exp1': ['A0_Baseline','A1_Channel','A2_Spatial','A3_ChannelSpatial','A4_Self','A5_Combined'],
              'exp2': ['B2_P2','B3_P3','B4_P4'],
              'exp3': ['C1_P2P3','C2_P3P4','C3_P2P3P4'],
              'exp4': ['D1_ChannelSpatial','D2_SpatialChannel'],
              'exp5': ['E1_Kernel3','E2_Kernel5','E3_Kernel7'],
              'additional': ['F1_SE','F2_ECA','F3_CBAM','F4_Coordinate']}
# Insertion position of each model (for the §23 Main Result Table)
MODEL_POSITION = {'A0_Baseline': '-', 'A1_Channel': 'P3', 'A2_Spatial': 'P3',
    'A3_ChannelSpatial': 'P3', 'A4_Self': 'P3', 'A5_Combined': 'P3',
    'B2_P2': 'P2', 'B3_P3': 'P3', 'B4_P4': 'P4',
    'C1_P2P3': 'P2+P3', 'C2_P3P4': 'P3+P4', 'C3_P2P3P4': 'P2+P3+P4',
    'D1_ChannelSpatial': 'P3', 'D2_SpatialChannel': 'P3',
    'E1_Kernel3': 'P3', 'E2_Kernel5': 'P3', 'E3_Kernel7': 'P3',
    'F1_SE': 'P3', 'F2_ECA': 'P3', 'F3_CBAM': 'P3', 'F4_Coordinate': 'P3'}
with open('results/data_splits.pkl', 'rb') as f:
    splits = pickle.load(f)
X_test, y_test = splits['X_test'], splits['y_test']
class_names = splits['class_names']
y_true = np.argmax(y_test, axis=1)
print(f"✓ Test set: {X_test.shape}, classes: {list(class_names)}")

In [ ]:
# Cell 2: Evaluate every trained model (accuracy, params, inference time)
from tensorflow import keras
rows = []
per_model_preds = {}
for exp, names in EXP_MODELS.items():
    for name in names:
        mp = f"results/models/{name}_model.h5"
        if not os.path.exists(mp):
            print(f"  ! {name}: weights not found, skipping (run Part 3 first)")
            continue
        model = keras.models.load_model(mp, compile=False)
        n_params = model.count_params()
        t0 = time.perf_counter()
        prob = model.predict(X_test, batch_size=32, verbose=0)
        infer_ms = (time.perf_counter() - t0) / len(X_test) * 1000.0
        pred = np.argmax(prob, axis=1)
        per_model_preds[name] = pred
        acc = accuracy_score(y_true, pred)
        prec, rec, f1, _ = precision_recall_fscore_support(y_true, pred, average='macro',
                                                          zero_division=0)
        prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_true, pred,
                                                              average='weighted', zero_division=0)
        rows.append({'experiment': exp, 'model': name,
                     'position': MODEL_POSITION[name],
                     'test_accuracy': acc,
                     'macro_precision': prec, 'macro_recall': rec, 'macro_f1': f1,
                     'weighted_f1': f1_w, 'params': n_params,
                     'infer_ms_per_sample': infer_ms})
        print(f"  ✓ {name}: acc={acc:.4f} macro-F1={f1:.4f} "
              f"params={n_params} infer={infer_ms:.2f}ms/sample")
metrics_df = pd.DataFrame(rows)

In [ ]:
# Cell 3: Build and save the 7 result tables
TABLES = {
    'table1_exp1_type': 'exp1', 'table2_exp2_position': 'exp2',
    'table3_exp3_multiposition': 'exp3', 'table4_exp4_order': 'exp4',
    'table5_exp5_kernel': 'exp5', 'table6_additional_mechanisms': 'additional'}
for tname, exp in TABLES.items():
    sub = metrics_df[metrics_df['experiment'] == exp].round(4)
    sub.to_csv(f"results/tables/{tname}.csv", index=False)
    print(f"\n{tname}:"); print(sub.to_string(index=False))
# Table 7: per-class report of the best model
best = metrics_df.loc[metrics_df['test_accuracy'].idxmax(), 'model']
rep = classification_report(y_true, per_model_preds[best], target_names=list(class_names),
                            output_dict=True, zero_division=0)
t7 = pd.DataFrame(rep).T.round(4)
t7.to_csv('results/tables/table7_best_perclass.csv')
print(f"\nBest model: {best}"); print(t7)
with open('results/all_metrics.pkl', 'wb') as f:
    pickle.dump({'metrics': metrics_df, 'best_model': best}, f)

In [ ]:
# Cell 4: Confusion matrices - baseline vs best model
import seaborn as sns
for name, fname in [('A0_Baseline', 'confusion_matrix_baseline'), (best, 'confusion_matrix_best')]:
    cm = confusion_matrix(y_true, per_model_preds[name])
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names,
                yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(f'Confusion Matrix - {name}')
    fig.tight_layout(); fig.savefig(f"results/figures/{fname}.png", dpi=150); plt.show()
print("✓ Confusion matrices saved")

In [ ]:
# Cell 5: Spec-format tables - §23 Main Result Table, §18 Kernel Table, §19.1 Mechanism Table
def row(name):
    r = metrics_df[metrics_df['model'] == name]
    return r.iloc[0] if len(r) else None
# §23 Main Result Table: Model | Position | Accuracy | Precision | Recall | F1 | Parameters | Inference Time
main_spec = [('A0_Baseline', 'Baseline', '-'), ('A1_Channel', 'Channel Attention', 'P3'),
    ('A2_Spatial', 'Spatial Attention', 'P3'), ('B2_P2', 'Channel+Spatial', 'P2'),
    ('A3_ChannelSpatial', 'Channel+Spatial', 'P3'), ('B4_P4', 'Channel+Spatial', 'P4'),
    ('A4_Self', 'Self-Attention', 'P3'), ('F2_ECA', 'Additional Attention-1 (ECA)', 'P3'),
    ('F3_CBAM', 'Additional Attention-2 (CBAM)', 'P3')]
main_rows = []
for name, label, pos in main_spec:
    r = row(name)
    if r is None:
        print(f"  ! {name} not evaluated yet"); continue
    main_rows.append({'Model': label, 'Position': pos, 'Accuracy': round(r['test_accuracy'], 4),
        'Precision': round(r['macro_precision'], 4), 'Recall': round(r['macro_recall'], 4),
        'F1': round(r['macro_f1'], 4), 'Parameters': int(r['params']),
        'InferenceTime_ms_per_sample': round(r['infer_ms_per_sample'], 3)})
main_df = pd.DataFrame(main_rows)
main_df.to_csv('results/tables/main_result_table.csv', index=False)
print("§23 Main Result Table:"); print(main_df.to_string(index=False))
# §18 Spatial Kernel Size Table: kernel | Accuracy | Macro F1 | Inference Time
kernel_spec = [('E1_Kernel3', '3x3'), ('E2_Kernel5', '5x5'), ('E3_Kernel7', '7x7')]
k_rows = []
for name, k in kernel_spec:
    r = row(name)
    if r is None: continue
    k_rows.append({'SpatialKernel': k, 'Accuracy': round(r['test_accuracy'], 4),
        'MacroF1': round(r['macro_f1'], 4),
        'InferenceTime_ms_per_sample': round(r['infer_ms_per_sample'], 3)})
pd.DataFrame(k_rows).to_csv('results/tables/kernel_size_table.csv', index=False)
print("\n§18 Kernel Table:"); print(pd.DataFrame(k_rows).to_string(index=False))
# §19.1 Attention Mechanism Comparison Table
mech_spec = [('A0_Baseline', 'No Attention'), ('F1_SE', 'Channel / SE'),
    ('A2_Spatial', 'Spatial Attention'), ('F2_ECA', 'ECA'), ('F3_CBAM', 'CBAM'),
    ('A4_Self', 'Self-Attention'), ('F4_Coordinate', 'Additional (Coordinate)')]
m_rows = []
for name, label in mech_spec:
    r = row(name)
    if r is None: continue
    m_rows.append({'Attention': label, 'Accuracy': round(r['test_accuracy'], 4),
        'Precision': round(r['macro_precision'], 4), 'Recall': round(r['macro_recall'], 4),
        'MacroF1': round(r['macro_f1'], 4), 'Parameters': int(r['params']),
        'InferenceTime_ms_per_sample': round(r['infer_ms_per_sample'], 3)})
pd.DataFrame(m_rows).to_csv('results/tables/mechanism_comparison_table.csv', index=False)
print("\n§19.1 Mechanism Table:"); print(pd.DataFrame(m_rows).to_string(index=False))

---

**Checklist Progress**:
- [x] Test accuracy / precision / recall / F1 for all models (+ params, inference time)
- [x] 7 per-experiment tables + §23 main_result_table.csv + §18 kernel_size_table.csv + §19.1 mechanism_comparison_table.csv
- [x] Confusion matrices (`confusion_matrix_baseline.png`, `confusion_matrix_best.png`)

**Next: Part 5 - Graphical Comparison Plots**